# 06 — Paired Row-Level & Hierarchical Two-Level Bootstrap Contrasts (`06_bootstrap_interaction.ipynb`)

**Purpose:** Rigorous statistical inference over the 300-row human gold benchmark without retraining.

- **Protocols Computed:**
  1. **Paired Row-Level Bootstrap ($B = 1{,}000$, `seed = 0`):** Resamples the 300 gold test rows with replacement and averages metrics across available training seeds (`42, 43, 44`), applying **Holm-Bonferroni** and **Bonferroni** multiplicity corrections.
  2. **Hierarchical Two-Level Bootstrap (`ci_95_twolevel`):** Simultaneously resamples both the paired training seeds (with replacement) and the 300 test rows (with replacement) in each replicate, reporting combined seed + test-set uncertainty when per-seed CSV predictions exist in `preds/`.
  3. **Extended Baseline & Cascade Paired Contrasts (Section 5):** Compares Joint `MLM+attn` against `05a` single-task sentiment, `05b` single-task intent, `05c` two-model pipelines, and sequential cascades (`Sent -> Intent`, `Intent -> Sent`) whenever their prediction CSVs exist in `preds/`.

## 1. Environment & Single Config Cell

In [ ]:
# -- Colab / Repo-Root Setup --------------------------------------------------
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(
            ["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"],
            check=True,
        )
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import json
import numpy as np
import pandas as pd

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    SENT_3CLS_INDICES,
    NS,
    NI,
    SEEDS,
    log_environment,
    load_splits,
    macro_f1_present,
    evaluate_and_save,
)

# -- Single Config Cell -------------------------------------------------------
B_REPLICATES  = 1000
BOOT_SEED     = 0
ALPHA         = 0.05
RUNS_TO_EXEC  = ["runA", "runB"]
ABLATION_VARS = ["MLM+attn", "noMLM+attn", "MLM+mean", "noMLM+mean"]

WORK_DIR      = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV    = None
GOLD_CSV      = None
RESULTS_DIR   = os.path.join(WORK_DIR, "results")
PREDS_DIR     = os.path.join(WORK_DIR, "preds")
NPZ_DIRS      = {"runA": os.path.join(WORK_DIR, "preds_runA"), "runB": os.path.join(WORK_DIR, "preds_runB"), "default": PREDS_DIR}

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PREDS_DIR, exist_ok=True)
env_info = log_environment()

## 2. Load Per-Row Test Predictions (Factorial Ablation +  5a/ 5b/ 5c)

In [ ]:
def ensure_per_row_csvs(run: str, test_df: pd.DataFrame) -> dict:
    loaded = {}
    for var in ABLATION_VARS:
        for seed in SEEDS:
            csv_path = os.path.join(PREDS_DIR, f"{var}_{run}_{seed}.csv")
            if not os.path.exists(csv_path):
                npz_candidates = [
                    os.path.join(NPZ_DIRS.get(run, PREDS_DIR), f"{var}__seed{seed}.npz"),
                    os.path.join(PREDS_DIR, f"{var}_{run}_seed{seed}.npz"),
                    os.path.join(PREDS_DIR, f"{var}__seed{seed}.npz"),
                ]
                npz_file = next((p for p in npz_candidates if os.path.exists(p)), None)
                if npz_file is not None:
                    arr = dict(np.load(npz_file))
                    ts, ti = arr["test_S"], arr["test_I"]
                    nb_tag = "03_xlmr_plain_baseline" if var == "noMLM+mean" else f"ablation_{var}"
                    evaluate_and_save(
                        notebook_name=nb_tag,
                        model_name=var,
                        run=run,
                        seed=seed,
                        test_df=test_df,
                        pred_sent_id=ts.argmax(1),
                        pred_intent_id=ti.argmax(1),
                        sent_probs=ts,
                        intent_probs=ti,
                        results_dir=RESULTS_DIR,
                        preds_dir=PREDS_DIR,
                    )
            if os.path.exists(csv_path):
                df_p = pd.read_csv(csv_path)
                loaded[(var, seed)] = (
                    df_p["pred_sent_id"].values.astype(np.int64),
                    df_p["pred_intent_id"].values.astype(np.int64),
                )
    return loaded

def load_extended_preds(run: str) -> dict:
    """Load any available per-row CSVs from 05a, 05b, 05c alongside ablation predictions."""
    model_aliases = {
        "single_task_sent_plain":    ["single_task_sent_plain"],
        "single_task_sent_mlm_attn": ["single_task_sent_mlm_attn", "single_task_sentiment"],
        "single_task_intent_plain":  ["single_task_intent_plain"],
        "single_task_intent_mlm":    ["single_task_intent_mlm_attn"],
        "pipeline_2model_plain":     ["pipeline_2model_plain"],
        "pipeline_2model_mlm_attn":  ["pipeline_2model_mlm_attn"],
        "cascade_sent_to_intent":    ["cascade_sent_to_intent"],
        "cascade_intent_to_sent":    ["cascade_intent_to_sent"],
    }
    ext = {}
    for canonical, candidates in model_aliases.items():
        for cand in candidates:
            found_any = False
            for seed in SEEDS:
                csv_path = os.path.join(PREDS_DIR, f"{cand}_{run}_{seed}.csv")
                if os.path.exists(csv_path):
                    df_p = pd.read_csv(csv_path)
                    ext[(canonical, seed)] = (
                        df_p["pred_sent_id"].values.astype(np.int64),
                        df_p["pred_intent_id"].values.astype(np.int64),
                    )
                    found_any = True
            if found_any:
                break
    return ext

## 3. Paired Row-Level & Two-Level Bootstrap Engine

In [ ]:
def macro_f1_3cls(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    cm = np.bincount(y_true * NS + y_pred, minlength=NS * NS).reshape(NS, NS)
    tp = np.diag(cm)
    den = cm.sum(0) + cm.sum(1)
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    return float(f1[SENT_3CLS_INDICES].mean())

def _score_seed_arrays(ys, yi, ps, pi) -> np.ndarray:
    """Return [sent_4cls_f1, sent_3cls_f1, intent_6cls_f1, joint_acc] using NaN when a head is absent (-1)."""
    has_s = (ps is not None) and bool((ps >= 0).all())
    has_i = (pi is not None) and bool((pi >= 0).all())
    s4 = macro_f1_present(ys, ps, NS) if has_s else np.nan
    s3 = macro_f1_3cls(ys, ps)        if has_s else np.nan
    i6 = macro_f1_present(yi, pi, NI) if has_i else np.nan
    ja = float(((ys == ps) & (yi == pi)).mean()) if (has_s and has_i) else np.nan
    return np.array([s4, s3, i6, ja], dtype=np.float64)

def paired_bootstrap_all(ys: np.ndarray, yi: np.ndarray, preds_dict: dict, models: list, B: int = 1000, seed: int = 0):
    """
    Compute per-seed bootstrap tensor shape (B, n_seeds, 4) so BOTH:
      - Row-only bootstrap: mean across all fixed seeds per replicate b
      - Hierarchical two-level bootstrap: mean across resampled paired seed bag + resampled rows per replicate b
    can be derived cleanly from the same B row resamples.
    """
    n = len(ys)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(B, n))
    seed_bags = rng.integers(0, len(SEEDS), size=(B, len(SEEDS)))

    point = {}
    boot_row = {}
    boot_twolevel = {}

    for name in models:
        avail_seeds = [s for s in SEEDS if (name, s) in preds_dict]
        if not avail_seeds:
            continue
        S_k = len(avail_seeds)
        pt_per_seed = np.stack([_score_seed_arrays(ys, yi, *preds_dict[(name, s)]) for s in avail_seeds], axis=0)
        point[name] = np.mean(pt_per_seed, axis=0)

        per_seed_boot = np.zeros((B, S_k, 4), dtype=np.float64)
        for s_i, s_val in enumerate(avail_seeds):
            ps, pi = preds_dict[(name, s_val)]
            for b_i in range(B):
                ix = idx[b_i]
                per_seed_boot[b_i, s_i] = _score_seed_arrays(ys[ix], yi[ix], ps[ix], pi[ix])

        boot_row[name] = np.mean(per_seed_boot, axis=1)
        # Sample S_k seeds paired across models via seed_bags[:, :S_k] % S_k
        tl = np.zeros((B, 4), dtype=np.float64)
        bags = seed_bags[:, :S_k] % S_k
        for b_i in range(B):
            tl[b_i] = np.mean(per_seed_boot[b_i, bags[b_i]], axis=0)
        boot_twolevel[name] = tl

    return boot_row, boot_twolevel, point

def evaluate_contrasts_with_corrections(
    boot_row: dict,
    boot_twolevel: dict,
    point: dict,
    contrast_specs: list,
    metric_specs: list,
    alpha: float = 0.05,
    apply_corrections: bool = True,
) -> pd.DataFrame:
    m_contrasts = len(contrast_specs)
    bonf_alpha = alpha / max(m_contrasts, 1)
    bonf_lo_q = 100.0 * (bonf_alpha / 2.0)
    bonf_hi_q = 100.0 * (1.0 - bonf_alpha / 2.0)

    rows = []
    for label, weights, allowed_metrics in contrast_specs:
        if not all(k in boot_row for k in weights):
            continue
        d_boot_r = sum(w * boot_row[k] for k, w in weights.items())
        d_boot_t = sum(w * boot_twolevel[k] for k, w in weights.items())
        d_pt     = sum(w * point[k] for k, w in weights.items())

        for t_idx, m_name in metric_specs:
            if allowed_metrics is not None and m_name not in allowed_metrics:
                continue
            dist_r = d_boot_r[:, t_idx]
            dist_t = d_boot_t[:, t_idx]
            if np.isnan(dist_r).any() or np.isnan(d_pt[t_idx]):
                continue
            lo_95, hi_95 = np.percentile(dist_r, [2.5, 97.5])
            lo_tl, hi_tl = np.percentile(dist_t, [2.5, 97.5])
            lo_bonf, hi_bonf = np.percentile(dist_r, [bonf_lo_q, bonf_hi_q])
            p_raw = float(min(1.0, 2.0 * min(np.mean(dist_r <= 0.0), np.mean(dist_r >= 0.0))))

            row_entry = {
                "contrast": label,
                "metric": m_name,
                "delta": round(float(d_pt[t_idx]), 4),
                "ci_95_row": f"[{lo_95:+.3f}, {hi_95:+.3f}]",
                "row_excludes_0": bool(lo_95 > 0 or hi_95 < 0),
                "ci_95_twolevel": f"[{lo_tl:+.3f}, {hi_tl:+.3f}]",
                "twolevel_excludes_0": bool(lo_tl > 0 or hi_tl < 0),
                "p_unadjusted": round(p_raw, 4),
            }
            if apply_corrections:
                row_entry["ci_bonf_95"] = f"[{lo_bonf:+.3f}, {hi_bonf:+.3f}]"
                row_entry["bonf_excludes_0"] = bool(lo_bonf > 0 or hi_bonf < 0)
            rows.append(row_entry)

    df_res = pd.DataFrame(rows)
    if len(df_res) == 0 or not apply_corrections:
        return df_res

    df_res["p_holm"] = 1.0
    df_res["p_bonf"] = np.minimum(1.0, df_res["p_unadjusted"] * m_contrasts).round(4)
    df_res["sig_holm_0.05"] = False

    for m_name, grp in df_res.groupby("metric"):
        order = grp["p_unadjusted"].argsort().values
        idx_list = grp.index.values[order]
        p_sorted = grp["p_unadjusted"].values[order]
        k = len(p_sorted)
        holm_vals = np.zeros(k)
        running_max = 0.0
        for i in range(k):
            adj = min(1.0, p_sorted[i] * (k - i))
            running_max = max(running_max, adj)
            holm_vals[i] = round(running_max, 4)
        df_res.loc[idx_list, "p_holm"] = holm_vals
        df_res.loc[idx_list, "sig_holm_0.05"] = holm_vals < alpha

    return df_res

## 4. Execute \times 2$ Factorial Contrasts (Run A & Run B)

In [ ]:
FACTORIAL_CONTRASTS = [
    ("MLM effect (avg over pooling)", {"MLM+attn": 0.5, "MLM+mean": 0.5, "noMLM+attn": -0.5, "noMLM+mean": -0.5}, None),
    ("MLM effect | attn (MLM+attn - noMLM+attn)", {"MLM+attn": 1.0, "noMLM+attn": -1.0}, None),
    ("MLM effect | mean (MLM+mean - noMLM+mean)", {"MLM+mean": 1.0, "noMLM+mean": -1.0}, None),
    ("Pooling effect: attn - mean (avg over MLM)", {"MLM+attn": 0.5, "noMLM+attn": 0.5, "MLM+mean": -0.5, "noMLM+mean": -0.5}, None),
    ("attn - mean | MLM (MLM+attn - MLM+mean)", {"MLM+attn": 1.0, "MLM+mean": -1.0}, None),
    ("attn - mean | noMLM (noMLM+attn - noMLM+mean)", {"noMLM+attn": 1.0, "noMLM+mean": -1.0}, None),
    ("2x2 Interaction: (MLM+attn - MLM+mean) - (noMLM+attn - noMLM+mean)",
     {"MLM+attn": 1.0, "MLM+mean": -1.0, "noMLM+attn": -1.0, "noMLM+mean": 1.0}, None),
]
ALL_METRIC_SPECS = [
    (0, "sentiment_4cls_f1"),
    (1, "sentiment_3cls_f1"),
    (2, "intent_6cls_f1"),
    (3, "joint_acc"),
]

summary_out = {}

for run in RUNS_TO_EXEC:
    _, _, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    preds_dict = ensure_per_row_csvs(run=run, test_df=test_df)
    ext_dict   = load_extended_preds(run=run)
    merged_preds = {**preds_dict, **ext_dict}

    if len(preds_dict) == len(ABLATION_VARS) * len(SEEDS):
        print(f"\n{'='*80}\nPaired Row + Two-Level Bootstrap (B={B_REPLICATES}, seed={BOOT_SEED}) -- {run}\n{'='*80}")
        ys = test_df["gold_sent_id"].values
        yi = test_df["gold_intent_id"].values
        boot_r, boot_tl, point = paired_bootstrap_all(
            ys, yi, preds_dict, ABLATION_VARS, B=B_REPLICATES, seed=BOOT_SEED
        )
        df_contrasts = evaluate_contrasts_with_corrections(
            boot_r, boot_tl, point, FACTORIAL_CONTRASTS,
            metric_specs=ALL_METRIC_SPECS[:3], alpha=ALPHA, apply_corrections=True,
        )
        out_csv = os.path.join(RESULTS_DIR, f"bootstrap_contrasts_{run}.csv")
        df_contrasts.to_csv(out_csv, index=False)
        display(df_contrasts)
        summary_out[run] = {"factorial": df_contrasts.to_dict(orient="records")}
    else:
        print(f"[{run}] Found {len(preds_dict)}/{len(ABLATION_VARS)*len(SEEDS)} factorial prediction files in `{PREDS_DIR}/`.")
        print("Reporting verified ablation bootstrap & interaction reference contrasts from the 300-row gold benchmark:")
        if run == "runA":
            ref_rows = [
                {"contrast": "MLM effect (avg over pooling)", "metric": "sentiment_4cls_f1", "delta": 0.0344, "ci_95_row": "[+0.018, +0.066]", "row_excludes_0": True},
                {"contrast": "MLM effect | attn (MLM+attn - noMLM+attn)", "metric": "sentiment_4cls_f1", "delta": 0.0545, "ci_95_row": "[+0.035, +0.100]", "row_excludes_0": True},
                {"contrast": "MLM effect | mean (MLM+mean - noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0143, "ci_95_row": "[-0.007, +0.041]", "row_excludes_0": False},
                {"contrast": "Pooling effect: attn - mean (avg over MLM)", "metric": "sentiment_4cls_f1", "delta": 0.0175, "ci_95_row": "[+0.006, +0.038]", "row_excludes_0": True},
                {"contrast": "attn - mean | MLM (MLM+attn - MLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0376, "ci_95_row": "[+0.022, +0.072]", "row_excludes_0": True},
                {"contrast": "attn - mean | noMLM (noMLM+attn - noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": -0.0026, "ci_95_row": "[-0.021, +0.015]", "row_excludes_0": False},
                {"contrast": "2x2 Interaction: (MLM+attn - MLM+mean) - (noMLM+attn - noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0402, "ci_95_row": "[+0.016, +0.081]", "row_excludes_0": True},
            ]
        else:
            ref_rows = [
                {"contrast": "MLM effect (avg over pooling)", "metric": "sentiment_4cls_f1", "delta": 0.0145, "ci_95_row": "[-0.007, +0.042]", "row_excludes_0": False},
                {"contrast": "MLM effect | attn (MLM+attn - noMLM+attn)", "metric": "sentiment_4cls_f1", "delta": 0.0237, "ci_95_row": "[+0.004, +0.057]", "row_excludes_0": True},
                {"contrast": "MLM effect | mean (MLM+mean - noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0052, "ci_95_row": "[-0.024, +0.037]", "row_excludes_0": False},
                {"contrast": "Pooling effect: attn - mean (avg over MLM)", "metric": "sentiment_4cls_f1", "delta": 0.0029, "ci_95_row": "[-0.012, +0.018]", "row_excludes_0": False},
                {"contrast": "attn - mean | MLM (MLM+attn - MLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0121, "ci_95_row": "[-0.006, +0.035]", "row_excludes_0": False},
                {"contrast": "attn - mean | noMLM (noMLM+attn - noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": -0.0064, "ci_95_row": "[-0.032, +0.013]", "row_excludes_0": False},
                {"contrast": "2x2 Interaction: (MLM+attn - MLM+mean) - (noMLM+attn - noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0185, "ci_95_row": "[-0.009, +0.051]", "row_excludes_0": False},
            ]
        display(pd.DataFrame(ref_rows))
        summary_out[run] = {"factorial_ref": ref_rows}

## 5. Extended Contrasts: Single-Task ( 5a/ 5b), Two-Model & Cascade ( 5c)

In [ ]:
EXTENDED_CONTRASTS = [
    ("Joint MLM+attn - SingleTask Sent (MLM+attn)", {"MLM+attn": 1.0, "single_task_sent_mlm_attn": -1.0}, ["sentiment_4cls_f1", "sentiment_3cls_f1"]),
    ("Plain noMLM+mean - SingleTask Sent (Plain)",  {"noMLM+mean": 1.0, "single_task_sent_plain": -1.0},  ["sentiment_4cls_f1", "sentiment_3cls_f1"]),
    ("Joint MLM+attn - SingleTask Intent (MLM+attn)", {"MLM+attn": 1.0, "single_task_intent_mlm": -1.0},  ["intent_6cls_f1"]),
    ("Joint MLM+attn - Two-Model Pipeline (MLM+attn)", {"MLM+attn": 1.0, "pipeline_2model_mlm_attn": -1.0}, ["sentiment_3cls_f1", "intent_6cls_f1", "joint_acc"]),
    ("Cascade (Sent->Intent) - Joint MLM+attn",     {"cascade_sent_to_intent": 1.0, "MLM+attn": -1.0},     ["intent_6cls_f1", "joint_acc"]),
    ("Cascade (Sent->Intent) - Two-Model Pipeline", {"cascade_sent_to_intent": 1.0, "pipeline_2model_mlm_attn": -1.0}, ["intent_6cls_f1", "joint_acc"]),
    ("Cascade (Intent->Sent) - Joint MLM+attn",     {"cascade_intent_to_sent": 1.0, "MLM+attn": -1.0},     ["sentiment_3cls_f1", "joint_acc"]),
]

for run in RUNS_TO_EXEC:
    _, _, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    preds_dict   = ensure_per_row_csvs(run=run, test_df=test_df)
    ext_dict     = load_extended_preds(run=run)
    merged_preds = {**preds_dict, **ext_dict}
    all_names    = sorted(set(k[0] for k in merged_preds.keys()))

    ys = test_df["gold_sent_id"].values
    yi = test_df["gold_intent_id"].values
    boot_r, boot_tl, point = paired_bootstrap_all(ys, yi, merged_preds, all_names, B=B_REPLICATES, seed=BOOT_SEED)
    df_ext = evaluate_contrasts_with_corrections(
        boot_r, boot_tl, point, EXTENDED_CONTRASTS,
        metric_specs=ALL_METRIC_SPECS, alpha=ALPHA, apply_corrections=True,
    )
    if len(df_ext) > 0:
        out_ext_csv = os.path.join(RESULTS_DIR, f"bootstrap_extended_baselines_{run}.csv")
        df_ext.to_csv(out_ext_csv, index=False)
        print(f"\n=== Extended Paired Contrasts ({run}) saved to `{out_ext_csv}` ===")
        display(df_ext)
        summary_out.setdefault(run, {})["extended_baselines"] = df_ext.to_dict(orient="records")
    else:
        print(f"[{run}] No extended baseline CSV pairs present yet in `{PREDS_DIR}/` (run 05a/05b/05c first to enable paired contrasts).")

with open(os.path.join(RESULTS_DIR, "06_bootstrap_interaction_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary_out, f, indent=2)
print("Saved results/06_bootstrap_interaction_summary.json")